# University Physics with Modern Physics

<img src="..//pics/cover.png" width=400> 



In [1]:
import setuptools

print("setuptools version:", setuptools.__version__)  # Should be < 70.0.0

import vpython as vp
from astropy import units as u
from astropy import constants as const
import sympy as sp
import numpy as np
from IPython.display import display, Math
import matplotlib.pyplot as plt
import sys
from pathlib import Path

print("All imports succeeded!")

setuptools version: 69.5.1


<IPython.core.display.Javascript object>

All imports succeeded!


In [2]:
# Add root folder to sys.path
sys.path.append(str(Path("..").resolve()))

# Import your helper function directly
from helper_functions import fmt_vec


In [3]:
!python --version

Python 3.12.10


In [4]:
pwd


'c:\\Users\\crodr\\BK_tech\\Physics\\BK_University_Physics_Young_15\\ch02'

# 2 Motion Along a Stright Line

## PROBLEMS Section 




#### 2.73 

•• A watermelon is dropped from the edge of the roof of a building
and falls to the ground. You are standing on the pavement and see
the watermelon falling when it is 30.0 m above the ground. Then 1.50 s
after you first spot it, the watermelon lands at your feet. What is the
height of the building? Neglect air resistance.

In [4]:
# Define symbolic variables
H, y1, dt, g = sp.symbols("H y1 dt g", positive=True)
v1 = sp.symbols("v1", real=True)

# 1. Equation for displacement during the spotted interval
eq_interval = sp.Eq(y1, v1 * dt + sp.Rational(1, 2) * g * dt**2)

# 2. Solve for v1
v1_expr = sp.solve(eq_interval, v1)[0]

# 3. Height fallen prior to being spotted: h_top = v1^2 / (2*g)
h_top_expr = v1_expr**2 / (2 * g)

# 4. Total building height H = h_top + y1
H_expr = sp.simplify(h_top_expr + y1)

display(Math(rf"v_1 = {sp.latex(v1_expr)}"))
display(Math(rf"H = {sp.latex(H_expr)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
# --- 1. Create Lambdified Function ---
calc_H = sp.lambdify((y1, dt, g), H_expr)
calc_v1 = sp.lambdify((y1, dt, g), v1_expr)

# --- 2. Astropy Physical Inputs ---
y1_qty = 30.0 * u.m
dt_qty = 1.50 * u.s
g_qty = const.g0

# --- 3. Evaluate Lambdified Functions ---
v1_val = calc_v1(y1_qty.value, dt_qty.value, g_qty.value)
v1_qty = v1_val * (u.m / u.s)

H_val = calc_H(y1_qty.value, dt_qty.value, g_qty.value)
H_qty = H_val * u.m

print(f"Speed when first spotted (v1) : {v1_qty:.2f}")
print(f"Total height of building (H)  : {H_qty:.2f}")

Speed when first spotted (v1) : 12.65 m / s
Total height of building (H)  : 38.15 m


#### 2.75

••• **Look Out Below**. Kemal heaves a 7.26 kg shot straight
up, giving it a constant upward acceleration from rest of 35.0 m/s2 for
64.0 cm. He releases it 2.20 m above the ground. Ignore air resistance.

**(a)** What is the speed of the shot when Kemal releases it? 

**(b)** How high above the ground does it go? 

**(c)** How much time does he have to get out
of its way before it returns to the height of the top of his head, 1.83 m
above the ground?

In [ ]:
# Define symbolic variables
a_push, dy_push, y_rel, y_head, g = sp.symbols(
    "a_{push} dy_{push} y_{rel} y_{head} g", positive=True
)
t_push, t_fall = sp.symbols("t_{push} t_{fall}", positive=True)

# (a) Release velocity v_rel
v_rel_expr = sp.sqrt(2 * a_push * dy_push)

# (b) Maximum height y_max
y_max_expr = y_rel + (v_rel_expr**2) / (2 * g)

# (c) Time calculation
# Push time
t_push_expr = sp.sqrt(2 * dy_push / a_push)

# Free fall displacement equation: y_head - y_rel = v_rel * t_fall - (1/2)*g*t_fall^2
dy_fall = y_head - y_rel
fall_eq = sp.Eq(dy_fall, v_rel_expr * t_fall - sp.Rational(1, 2) * g * t_fall**2)
t_fall_expr = sp.solve(fall_eq, t_fall)[1]  # Positive root

t_total_expr = t_push_expr + t_fall_expr

display(Math(rf"(a)\ v_{{\mathrm{{rel}}}} = {sp.latex(v_rel_expr)}"))
display(Math(rf"(b)\ y_{{\mathrm{{max}}}} = {sp.latex(sp.simplify(y_max_expr))}"))
display(Math(rf"(c)\ t_{{\mathrm{{total}}}} = {sp.latex(t_total_expr)}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# --- 1. Create Lambdified Functions ---
calc_v_rel = sp.lambdify((a_push, dy_push), v_rel_expr)
calc_y_max = sp.lambdify((a_push, dy_push, y_rel, g), y_max_expr)
calc_t_total = sp.lambdify((a_push, dy_push, y_rel, y_head, g), t_total_expr)

# --- 2. Astropy Physical Inputs ---
a_push_qty = 35.0 * (u.m / u.s**2)
dy_push_qty = 0.640 * u.m
y_rel_qty = 2.20 * u.m
y_head_qty = 1.83 * u.m
g_qty = const.g0

# --- 3. Evaluate Lambdified Functions ---
v_rel_val = calc_v_rel(a_push_qty.value, dy_push_qty.value)
v_rel_qty = v_rel_val * (u.m / u.s)

y_max_val = calc_y_max(
    a_push_qty.value, dy_push_qty.value, y_rel_qty.value, g_qty.value
)
y_max_qty = y_max_val * u.m

t_total_val = calc_t_total(
    a_push_qty.value,
    dy_push_qty.value,
    y_rel_qty.value,
    y_head_qty.value,
    g_qty.value,
)
t_total_qty = t_total_val * u.s

print(f"(a) Release speed (v_rel)     : {v_rel_qty:.2f}")
print(f"(b) Maximum height (y_max)     : {y_max_qty:.2f}")
print(f"(c) Total available time (t)   : {t_total_qty:.2f}")

(a) Release speed (v_rel)     : 6.69 m / s
(b) Maximum height (y_max)     : 4.48 m
(c) Total available time (t)   : 1.61 s


#### 2.76
**Multistage Rocket**. In the first stage of a two-stage
rocket, the rocket is fired from the launch pad starting from rest but
with a constant acceleration of 3.50 m/s2 upward. At 25.0 s after launch,
the second stage fires for 10.0 s, which boosts the rocket’s velocity to
132.5 m/s upward at 35.0 s after launch. This firing uses up all of the
fuel, however, so after the second stage has finished firing, the only
force acting on the rocket is gravity. Ignore air resistance. 

**(a)** Find the
maximum height that the stage-two rocket reaches above the launch
pad. 

**(b)** How much time after the end of the stage-two firing will it take
for the rocket to fall back to the launch pad? 

**(c)** How fast will the stagetwo
rocket be moving just as it reaches the launch pad?

In [ ]:
# Mathematical Solution
# Given Data  Stage 1
t0 = 0 * u.s
t1 = 25.0 * u.s
y0 = 0 * u.m  # initial position
v0 = 0 * (u.m / u.s)
a1 = 3.50 * (u.m / u.s**2)
dt1 = 25.0 * u.s

# Calculation end of stage 1
# v = v0 + a * t
v1 = v0 + a1 * dt1
# y = y0 + v0  t + 1/2 a t^2
y1 = (1 / 2) * a1 * (dt1) ** 2
print("Stage 1:")
print(f"Velocity: (v1) = {v1:.3g}")
print(f"Height: (y1) = {y1:.2f}")

# Given Data  Stage 2
t2 = 35.0 * u.s
dt2 = 10.0 * u.s
v2 = 132.5 * (u.m / u.s)

# Calculation end of stage 2
# a = Delta v / Delta t
a2 = (v2 - v1) / dt2

# y = + v0  t + 1/2 a t^2
dy2 = v1 * dt2 + (1 / 2) * a2 * dt2**2
y2 = y1 + dy2


print("Stage 2:")
print(f"Acceleration: (a2) = {a2:.2f}")
print(f"Delta y2: (dy2) = {dy2:.2f}")
print(f"Height at end stage 2: (y2) = {y2:.2f}")


# Given Data  Stage 3
g = -const.g0

print("Stage 3:")
print(f"Rocket is in a free fall affecty by gravity: (g) = {g:.2f}")


# (a) Maximum height above launch pad (y_max)
# Apex v = 0
# v^2 = v0^2 + 2gy ==> y = 0 - v02 / 2g
dy3 = -(v2**2) / (2 * g)
y_max = y2 + dy3

print("\nPart (a):")
print(f"Delta y3: (dy3) = {dy3:.2f}")
print(f"Maximun height above launch pod: (y_max) = {y_max:.2f}")


Stage 1:
Velocity: (v1) = 87.5 m / s
Height: (y1) = 1093.75 m
Stage 2:
Acceleration: (a2) = 4.50 m / s2
Delta y2: (dy2) = 1100.00 m
Height at end stage 2: (y2) = 2193.75 m
Stage 3:
Rocket is in a free fall affecty by gravity: (g) = -9.81 m / s2

Part (a):
Delta y3: (dy3) = 895.12 m
Maximun height above launch pod: (y_max) = 3088.87 m


In [ ]:
# Define symbolic variables
a1, dt1, dt2, v2, g = sp.symbols("a1 dt1 dt2 v2 g", positive=True)
dt_fall = sp.symbols("dt_{fall}", positive=True)

# Stage 1
v1_expr = a1 * dt1
y1_expr = sp.Rational(1, 2) * a1 * dt1**2

# Stage 2 (Constant acceleration assumption)
a2_expr = (v2 - v1_expr) / dt2
dy2_expr = v1_expr * dt2 + sp.Rational(1, 2) * a2_expr * dt2**2
y2_expr = sp.simplify(y1_expr + dy2_expr)

# (a) Maximum height y_max
dy3_expr = v2**2 / (2 * g)
ymax_expr = sp.simplify(y2_expr + dy3_expr)

# (b) Time after stage 2 end to reach ground
# Equation: -y2 = v2 * dt_fall - 1/2 * g * dt_fall^2
fall_eq = sp.Eq(-y2_expr, v2 * dt_fall - sp.Rational(1, 2) * g * dt_fall**2)
dt_fall_expr = sp.solve(fall_eq, dt_fall)[1]  # Positive root

# (c) Impact speed
v_impact_expr = sp.sqrt(v2**2 + 2 * g * y2_expr)

display(Math(rf"(a)\ y_{{\mathrm{{max}}}} = {sp.latex(ymax_expr)}"))
display(Math(rf"(b)\ \Delta t_{{\mathrm{{fall}}}} = {sp.latex(dt_fall_expr)}"))
display(Math(rf"(c)\ v_{{\mathrm{{impact}}}} = {sp.latex(v_impact_expr)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [312]:
# --- 1. Create Lambdified Functions ---
calc_ymax = sp.lambdify((a1, dt1, dt2, v2, g), ymax_expr)
calc_dt_fall = sp.lambdify((a1, dt1, dt2, v2, g), dt_fall_expr)
calc_v_impact = sp.lambdify((a1, dt1, dt2, v2, g), v_impact_expr)

# --- 2. Astropy Physical Inputs ---
a1_qty = 3.50 * (u.m / u.s**2)
dt1_qty = 25.0 * u.s
dt2_qty = 10.0 * u.s
v2_qty = 132.5 * (u.m / u.s)
g_qty = const.g0

# --- 3. Evaluate Lambdified Functions ---
ymax_val = calc_ymax(
    a1_qty.value, dt1_qty.value, dt2_qty.value, v2_qty.value, g_qty.value
)
ymax_qty = ymax_val * u.m

dt_fall_val = calc_dt_fall(
    a1_qty.value, dt1_qty.value, dt2_qty.value, v2_qty.value, g_qty.value
)
dt_fall_qty = dt_fall_val * u.s

v_impact_val = calc_v_impact(
    a1_qty.value, dt1_qty.value, dt2_qty.value, v2_qty.value, g_qty.value
)
v_impact_qty = v_impact_val * (u.m / u.s)

print(f"(a) Maximum height (y_max)     : {ymax_qty:.2f} ({ymax_qty.to(u.km):.2f})")
print(f"(b) Time after stage 2 firing  : {dt_fall_qty:.2f}")
print(f"(c) Impact speed (v_impact)    : {v_impact_qty:.2f}")

(a) Maximum height (y_max)     : 3088.87 m (3.09 km)
(b) Time after stage 2 firing  : 38.61 s
(c) Impact speed (v_impact)    : 246.14 m / s


#### 2.78 

page 92

••• During your summer internship for an aerospace company, you
are asked to design a small research rocket. The rocket is to be launched
from rest from the earth’s surface and is to reach a maximum height of
960 m above the earth’s surface. The rocket’s engines give the rocket an
upward acceleration of 16.0 m/s2 during the time T that they fire. After the
engines shut off, the rocket is in free fall. Ignore air resistance. What must
be the value of T in order for the rocket to reach the required altitude?

In [ ]:
# Define symbolic variables
T, a1, a_y, y_max = sp.symbols("T a1 a_y y_max")

# Phase 1: Cutoff velocity and position
v1 = a1 * T
y1 = sp.Rational(1, 2) * a1 * T**2

# Phase 2: Apex equation using explicit free-fall acceleration a_y (where a_y < 0)
# v_apex^2 = v1^2 + 2 * a_y * (y_max - y1) where v_apex = 0
apex_eq = sp.Eq(0, v1**2 + 2 * a_y * (y_max - y1))

# Solve for positive T assuming a1 > 0, y_max > 0, and a_y < 0
T_sols = sp.solve(apex_eq, T)
T_sol = sp.simplify(T_sols[1])  # Select the positive root

display(Math(rf"v_1 = {sp.latex(v1)}"))
display(Math(rf"y_1 = {sp.latex(y1)}"))
display(Math(rf"T = {sp.latex(T_sol)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# --- 1. Create Lambdified Function ---
# Function accepts a_y explicitly so we can pass a negative acceleration value
calc_T = sp.lambdify((a1, a_y, y_max), T_sol)

# --- 2. Astropy Physical Inputs ---
a1_qty = 16.0 * (u.m / u.s**2)
y_max_qty = 960.0 * u.m

# Explicitly define negative gravitational acceleration vector in astropy
a_y_qty = -const.g0  # -9.80665 m / s^2

# --- 3. Evaluate Lambdified Function ---
T_qty = calc_T(a1_qty, a_y_qty, y_max_qty)

# Derive cutoff values for reference
y1_qty = 0.5 * a1_qty * T_qty**2
v1_qty = a1_qty * T_qty

print(f"Free-fall acceleration (a_y)  : {a_y_qty:.2f}")
print(f"Required engine burn time (T) : {T_qty:.2f}")
print(f"Height at cutoff (y1)         : {y1_qty:.2f}")
print(f"Velocity at cutoff (v1)       : {v1_qty:.2f}")

Free-fall acceleration (a_y)  : -9.81 m / s2
Required engine burn time (T) : 6.75 s
Height at cutoff (y1)         : 364.80 m
Velocity at cutoff (v1)       : 108.05 m / s


#### 2.79 

page 92

••• A helicopter carrying Dr. Evil takes off with a constant upward
acceleration of 5.0 m/s2. Secret agent Austin Powers jumps on
just as the helicopter lifts off the ground. After the two men struggle
for 10.0 s, Powers shuts off the engine and steps out of the helicopter.
Assume that the helicopter is in free fall after its engine is shut off,
and ignore the effects of air resistance. 

**(a)** What is the maximum height
above ground reached by the helicopter? 

**(b)** Powers deploys a jet pack
strapped on his back 7.0 s after leaving the helicopter, and then he has
a constant downward acceleration with magnitude 2.0 m/s2. How far is
Powers above the ground when the helicopter crashes into the ground?

In [ ]:
# Symbolic variables
a1, t1, a_y, y_max, t_crash = sp.symbols("a1 t1 a_y y_max t_crash")
t_jet_start, a_jet = sp.symbols("t_jet_start a_jet")

# Phase 1: Powered ascent until cutoff
v1 = a1 * t1
y1 = sp.Rational(1, 2) * a1 * t1**2

# (a) Helicopter max height (v_apex = 0)
# 0 = v1^2 + 2 * a_y * (y_max - y1)
ymax_eq = sp.Eq(0, v1**2 + 2 * a_y * (y_max - y1))
ymax_expr = sp.solve(ymax_eq, y_max)[0]

# (b) Helicopter crash time (y = 0)
# 0 = y1 + v1 * t_crash + 1/2 * a_y * t_crash^2
crash_eq = sp.Eq(0, y1 + v1 * t_crash + sp.Rational(1, 2) * a_y * t_crash**2)
t_crash_sols = sp.solve(crash_eq, t_crash)
# Positive root selected symbolically assuming a_y < 0 and v1 > 0
t_crash_expr = t_crash_sols[1]

# Powers' motion:
# Free fall phase from t_rel = 0 to t_jet_start
y_P_start = y1 + v1 * t_jet_start + sp.Rational(1, 2) * a_y * t_jet_start**2
v_P_start = v1 + a_y * t_jet_start

# Jet pack phase from t_jet_start to t_crash
dt_jet = t_crash - t_jet_start
y_P_crash_expr = y_P_start + v_P_start * dt_jet + sp.Rational(1, 2) * a_jet * dt_jet**2

display(Math(rf"v_1 = {sp.latex(v1)}"))
display(Math(rf"y_1 = {sp.latex(y1)}"))
display(Math(rf"(a)\ y_{{\mathrm{{max}}}} = {sp.latex(ymax_expr)}"))
display(Math(rf"(b)\ t_{{\mathrm{{crash}}}} = {sp.latex(t_crash_expr)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# --- 1. Create Lambdified Functions ---
calc_v1 = sp.lambdify((a1, t1), v1)
calc_y1 = sp.lambdify((a1, t1), y1)
calc_ymax = sp.lambdify((a1, t1, a_y), ymax_expr)
calc_t_crash = sp.lambdify((a1, t1, a_y), t_crash_expr)
calc_y_P_crash = sp.lambdify((a1, t1, a_y, t_jet_start, a_jet, t_crash), y_P_crash_expr)

# --- 2. Astropy Physical Inputs ---
a1_qty = 5.0 * (u.m / u.s**2)
t1_qty = 10.0 * u.s
a_y_qty = -g0  # -9.80665 m/s^2 (explicit negative)
t_jet_start_qty = 7.0 * u.s
a_jet_qty = -2.0 * (u.m / u.s**2)  # Downward constant acceleration

# --- 3. Numerical Evaluations ---
v1_val = calc_v1(a1_qty.value, t1_qty.value)
v1_qty = v1_val * (u.m / u.s)

y1_val = calc_y1(a1_qty.value, t1_qty.value)
y1_qty = y1_val * u.m

ymax_val = calc_ymax(a1_qty.value, t1_qty.value, a_y_qty.value)
ymax_qty = ymax_val * u.m

t_crash_val = calc_t_crash(a1_qty.value, t1_qty.value, a_y_qty.value)
t_crash_qty = t_crash_val * u.s

y_P_crash_val = calc_y_P_crash(
    a1_qty.value,
    t1_qty.value,
    a_y_qty.value,
    t_jet_start_qty.value,
    a_jet_qty.value,
    t_crash_qty.value,
)
y_P_crash_qty = y_P_crash_val * u.m

print(f"Helicopter velocity at cutoff (v1) : {v1_qty:.2f}")
print(f"Helicopter height at cutoff (y1)   : {y1_qty:.2f}")
print(f"(a) Maximum height reached (ymax)  : {ymax_qty:.2f}")
print(f"    Helicopter crash time post-cutoff: {t_crash_qty:.2f}")
print(f"(b) Powers height at crash time    : {y_P_crash_qty:.2f}")

Helicopter velocity at cutoff (v1) : 50.00 m / s
Helicopter height at cutoff (y1)   : 250.00 m
(a) Maximum height reached (ymax)  : 377.46 m
    Helicopter crash time post-cutoff: 13.87 s
(b) Powers height at crash time    : 184.36 m


#### 2.80 

•• **Cliff Height**. You are climbing in the Altai when you suddenly
find yourself at the edge of a fog-shrouded cliff. To find the height
of this cliff, you drop a rock from the top; 8.00 s later you hear the sound
of the rock hitting the ground at the foot of the cliff. 

**(a)** If you ignore
air resistance, how high is the cliff if the speed of sound is 330 m>s ?

**(b)** Suppose you had ignored the time it takes the sound to reach you. In
that case, would you have overestimated or underestimated the height of
the cliff? Explain.

In [ ]:
h, g, t_fall, v_sound, t_sound, T = sp.symbols(
    "h, g, t_fall, v_sound, t_sound, T ", positive=True, real=True
)
h_fall_eq = sp.Eq(h, sp.Rational(1, 2) * g * t_fall**2)
t_fall_expr = sp.solve(h_fall_eq, t_fall)[0]
h_approx_expr = sp.Rational(1, 2) * g * t_fall**2

h_sound_eq = sp.Eq(h, v_sound * t_sound)
t_sound_expr = sp.solve(h_sound_eq, t_sound)[0]

T_eq = sp.Eq(T, t_fall_expr + t_sound_expr)
h_expr = sp.solve(T_eq, h)[0]

display(Math(rf"t_{{fall}} = {sp.latex(t_fall_expr)}"))
display(Math(rf"t_{{sound}} = {sp.latex(t_sound_expr)}"))
display(Math(rf"T = {sp.latex(T_expr)}"))
display(Math(rf"h = {sp.latex(h_expr)}"))
display(Math(rf"h_{{approx}} = {sp.latex(h_approx_expr)}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
calc_h = sp.lambdify((v_sound, T, g), h_expr)
calc_h_approx = sp.lambdify((g, t_fall), h_approx_expr)

v_sound = 330 * (u.m / u.s)
T_qty = 8 * u.s
g_qty = const.g0
# g_qty = 9.80 * (u.m/u.s**2)

h_qty = calc_h(v_sound, T_qty, g_qty)
print(f"Part (a) Exact cliff height (h): {h_qty:.2f}")

# Part (b) if you ignore the tavel time of sound, full T is for rock falling.
h_approx_qty = calc_h_approx(g_qty, T_qty)
print(f"Part (b) Approximate height (ignoring sound): {h_approx_qty:.2f}")
print(f"         Overstimation error: {(h_approx_qty - h_qty):.2f}")


Part (a) Exact cliff height (h): 255.92 m
Part (b) Approximate height (ignoring sound): 313.81 m
         Overstimation error: 57.89 m
